# Create Dust_temp.in files
Store each file in its respective climate input directory on s3, which means it'll get copied over to the run directory. (means we need to have already created the climate input dir!)

## Read in dust flux timeseries by site 

In [21]:
# --- where to save outputs ----------
#
#  ----------------------------------------------------------------
# [ RUN NAMES ]
# exe_central1p5 ; exe_central2p0 ; exe_limBio1p5 ; exe_lowCCS1p5 ; 
# exe_lowCDR1p5 ; exe_noBiochar1p5 ; exe_noWeathering1p5 ; 
# exe_ref1p5 ; exe_techBehaviorChange1p5 ; exe_techBehaviorChange2p0 ;
# 
# (updated dust)
# exe_central1p5_max10tha   exe_noWeathering1p5_max5tha
# ----------------------------------------------------------------
#
# <Site>_<deg_resolution>_<climtype>
RUNNAME = "exe_noWeathering1p5"
RUN_SUFFIX = "_max5tha"
climpath_main = f"s3://carbonplan-carbon-removal/ew-workflows-data/scepter/clim/one-cell-per-region_1deg/{RUNNAME}{RUN_SUFFIX}"
climtype = "yearly_ltm"
# ------------------------------------

In [22]:
import math
import os 
from typing import Optional, List

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr 

from ew_workflows import scepter_helperFxns as shf

# --- read in data
deg_resolution = 1
datpath = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated'
subdir = f'one-cell-per-region/{RUNNAME}'
fn_df = f'annual_rockflx_densest-ag-region_{deg_resolution}deg.pkl'
fn_ds = f'cropland+gcamRegions_time_{deg_resolution}deg.zarr'

# dataframe 
df = pd.read_pickle(os.path.join(datpath, subdir, fn_df))
df

,reg_id,region,time,rockflx_t_ha,rockflx_Mt_ha,rockflx_Mt,lat,lon,gcam_gross_cdr_mtCO2,region_nospaces
0,1,USA,2020,0.000000,0.000000e+00,0.000000,40.5,-97.5,0.000000,USA
1,1,USA,2021,0.001297,1.296965e-09,0.001054,40.5,-97.5,0.061658,USA
2,1,USA,2022,0.002594,2.593931e-09,0.002108,40.5,-97.5,0.123315,USA
3,1,USA,2023,0.003891,3.890896e-09,0.003161,40.5,-97.5,0.184973,USA
4,1,USA,2024,0.005188,5.187861e-09,0.004215,40.5,-97.5,0.246630,USA
...,...,...,...,...,...,...,...,...,...,...
2587,32,Colombia,2096,17.490362,1.749036e-05,2.485780,3.5,-76.5,24.752043,Colombia
2588,32,Colombia,2097,17.540740,1.754074e-05,2.492940,3.5,-76.5,24.823336,Colombia
2589,32,Colombia,2098,17.591117,1.759112e-05,2.500100,3.5,-76.5,24.894630,Colombia
2590,32,Colombia,2099,17.641495,1.764149e-05,2.507259,3.5,-76.5,24.965923,Colombia


In [ ]:
# # --- make any modifications
# # -----------------------------------------------------------------
# # [ MAX 10 T/HA ]
# # df["rockflx_t_ha"] = df["rockflx_t_ha"].clip(upper=10)
# # df["rockflx_Mt_ha"] = df["rockflx_Mt_ha"].clip(upper=10e-6)
# # df["rockflx_Mt"] = np.nan

# # [ MAX 5 T/HA ]
# df["rockflx_t_ha"] = df["rockflx_t_ha"].clip(upper=5)
# df["rockflx_Mt_ha"] = df["rockflx_Mt_ha"].clip(upper=5e-6)
# df["rockflx_Mt"] = np.nan

# # [ ROCK AMOUNT MULTIPLIER ]
# rock_multiplier = 2
# df["rockflx_t_ha"] = df["rockflx_t_ha"] * rock_multiplier
# df["rockflx_Mt_ha"] = df["rockflx_Mt_ha"] * rock_multiplier
# df["rockflx_Mt"] = np.nan
# # -----------------------------------------------------------------
# df

In [25]:
# ---------------------------------------
# [ set up function inputs ]
all_regions = df['region_nospaces'].unique()
t_add = 0.1
taudust = 0.05
output_filename = "Dust_temp.in"
dryrun = False
# ---------------------------------------

for this_region in all_regions:
        print(this_region)
        # pull out this region 
        dfreg = df[df['region'] == this_region].copy()
        # create output dir
        outdir = os.path.join(climpath_main, f'{this_region}_{deg_resolution}_{climtype}')
        climpath = outdir # read and write to same dir

        # create Dust_temp.in file
        outdict = shf.create_dust_input_from_gcam_df(
                outdir,
                climpath,
                df = dfreg,
                t_add = t_add,
                taudust = taudust,
                output_filename = "Dust_temp.in",
                dryrun = dryrun,
        )


USA
Africa_Eastern
Africa_Northern
Africa_Southern
Africa_Western
Australia_NZ
Brazil
Canada
Central_America_and_Caribbean
Central_Asia
China
EU-12
EU-15
Europe_Eastern
Europe_Non_EU
European_Free_Trade_Association
India
Indonesia
Japan
Mexico
Middle_East
Pakistan
Russia
South_Africa
South_America_Northern
South_America_Southern
South_Asia
South_Korea
Southeast_Asia
Taiwan
Argentina
Colombia


In [6]:
# plt.plot(outdict['time_yr'], outdict['cumrate_gm2yr'])
# plt.plot(dfreg['time_yr'], dfreg['rockflx_t_ha'].cumsum()*100)
# plt.plot(outdict['time_yr'], outdict['rate_gm2yr'])
# plt.xlim(70,80)
# plt.ylim(7e4,8.7e4)

In [20]:
# ---